# Python, Лекция 11.

[Оригинал](https://github.com/DanielShinoda/ami_python_25_lectures/blob/main/lectures/11.%20Namespaces.%20%D0%94%D0%B5%D0%BA%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D1%8B/Lecture_11.ipynb) лекции.

Итак, сегодня мы поговорим про простанства имен и декораторы

### [Namespaces](https://docs.python.org/3/tutorial/classes.html#python-scopes-and-namespaces)

Предыдущее занятие о функциях и классах дало интуитивное понимание локальных и глобальных переменных. Сегодня формализуем понятие "пространство имен" и механизмы разрешения имен в Python.

Под **пространством имен** (`namespace`) будем понимать отображение имя -> объект.

**Область видимости** (`scope`) — это текстовая область программы на Python, в пределах которой пространство имён доступно напрямую. "Доступно напрямую" означает, что обращение к имени пытается найти это имя в данном пространстве имён.


In [1]:
# начало scope
# global namespace
x = "модуль"


def f() -> None:
    # начало scope
    x = "локальная"
    # достали x из local namespace
    print("в f:", x)
    # конец scope


# f создает новый local namespace при вызове
f()
# конец scope

в f: локальная


Блоки `if`/`for`/`while`/`with` новых областей видимости не создают; функция (и `lambda`) при вызове — создает; `comprehension`/генераторные выражения в `Python 3` создают свой маленький локальный `scope` для переменной цикла.

Что будет?

In [2]:
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])

[2, 2, 2]


lambda захватывает не значение i, а саму переменную i (ячейку) из enclosing‑scope comprehension.

<div style="
    background-color: #FFBA00;
    padding: 15px;
    border-left: 5px solid #ffcc00;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Лучше не использовать lambda, обсуждали в конце лекции "Ссылки. Изменяемость."
    </span>
</div>

Python ищет имя по порядку `LEGB`: сначала в текущей локальной области (`L`, `Local`), потом во внешних функциях (`E`, `Enclosing`), потом в модуле (`G`, `Global`), потом во встроенных именах (`B`, `Builtins`).

![image](https://github.com/DanielShinoda/ami_python_25_lectures/blob/main/lectures/11.%20Namespaces.%20%D0%94%D0%B5%D0%BA%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D1%8B/legb.png?raw=true)


- `Global`: содержит `funcs`
- `Enclosing` (охватывающее для `lambda`): каждое `list‑comprehension` в Python исполняется в своём локальном `scope`; переменная `i` — локальная именно для этого скрытого "функционального" `scope comprehension`.
- `Local` (у `lambda`): появляется только при вызове `lambda`; внутри тела `lambda` нет собственного `i`, поэтому поиск идёт во внешнее (`enclosing`) пространство.
- `Builtins`: здесь находятся `print` и `range`

Фикс через аргумент по умолчанию:

In [3]:
funcs = [lambda i=i: i for i in range(3)]
print([f() for f in funcs])

[0, 1, 2]


Мы начали с необычного примера, чтобы интуитивно увидеть: имя ищется по цепочке областей видимости, а замыкания захватывают не значение, а ячейку имени. Теперь разберём формально каждое звено этой цепочки.

`builtins namespace`:

- Набор "встроенных" имён: `print`, `range`, `len`, `int`, `str`, `Exception`, `True`/`False`/`None`, ...
- Cоздаётся при инициализации интерпретатора; один на интерпретатор.
- Как используется: последнее звено в `LEGB`; имена доступны без импорта.

In [4]:
print(dir(__builtins__))

['ArithmeticError', 'AssertionError', 'AttributeError', 'BaseException', 'BaseExceptionGroup', 'BlockingIOError', 'BrokenPipeError', 'BufferError', 'BytesWarning', 'ChildProcessError', 'ConnectionAbortedError', 'ConnectionError', 'ConnectionRefusedError', 'ConnectionResetError', 'DeprecationWarning', 'EOFError', 'Ellipsis', 'EncodingWarning', 'EnvironmentError', 'Exception', 'ExceptionGroup', 'False', 'FileExistsError', 'FileNotFoundError', 'FloatingPointError', 'FutureWarning', 'GeneratorExit', 'IOError', 'ImportError', 'ImportWarning', 'IndentationError', 'IndexError', 'InterruptedError', 'IsADirectoryError', 'KeyError', 'KeyboardInterrupt', 'LookupError', 'MemoryError', 'ModuleNotFoundError', 'NameError', 'None', 'NotADirectoryError', 'NotImplemented', 'NotImplementedError', 'OSError', 'OverflowError', 'PendingDeprecationWarning', 'PermissionError', 'ProcessLookupError', 'PythonFinalizationError', 'RecursionError', 'ReferenceError', 'ResourceWarning', 'RuntimeError', 'RuntimeWarning

In [5]:
import builtins

builtins.print(dir(builtins))

['ArithmeticError', 'AssertionError', 'AttributeError', 'BaseException', 'BaseExceptionGroup', 'BlockingIOError', 'BrokenPipeError', 'BufferError', 'BytesWarning', 'ChildProcessError', 'ConnectionAbortedError', 'ConnectionError', 'ConnectionRefusedError', 'ConnectionResetError', 'DeprecationWarning', 'EOFError', 'Ellipsis', 'EncodingWarning', 'EnvironmentError', 'Exception', 'ExceptionGroup', 'False', 'FileExistsError', 'FileNotFoundError', 'FloatingPointError', 'FutureWarning', 'GeneratorExit', 'IOError', 'ImportError', 'ImportWarning', 'IndentationError', 'IndexError', 'InterruptedError', 'IsADirectoryError', 'KeyError', 'KeyboardInterrupt', 'LookupError', 'MemoryError', 'ModuleNotFoundError', 'NameError', 'None', 'NotADirectoryError', 'NotImplemented', 'NotImplementedError', 'OSError', 'OverflowError', 'PendingDeprecationWarning', 'PermissionError', 'ProcessLookupError', 'PythonFinalizationError', 'RecursionError', 'ReferenceError', 'ResourceWarning', 'RuntimeError', 'RuntimeWarning

`global namespace`:
- Что это: словарь `globals()` конкретного модуля (включая `__name__`, `__builtins__`, ваши функции/классы/переменные).
- Когда создаётся: при выполнении модуля (импорт или запуск).
- Время жизни: пока живёт модуль (обычно до завершения интерпретатора).


In [6]:
globals()

{'__name__': '__main__',
 '__doc__': 'Automatically created module for IPython interactive environment',
 '__package__': None,
 '__loader__': None,
 '__spec__': None,
 '__builtin__': <module 'builtins' (built-in)>,
 '__builtins__': <module 'builtins' (built-in)>,
 '_ih': ['',
  '# начало scope\n# global namespace\nx = "модуль"\n\n\ndef f() -> None:\n    # начало scope\n    x = "локальная"\n    # достали x из local namespace\n    print("в f:", x)\n    # конец scope\n\n\n# f создает новый local namespace при вызове\nf()\n# конец scope',
  'funcs = [lambda: i for i in range(3)]\nprint([f() for f in funcs])',
  'funcs = [lambda i=i: i for i in range(3)]\nprint([f() for f in funcs])',
  'print(dir(__builtins__))',
  'import builtins\n\nbuiltins.print(dir(builtins))',
  'globals()'],
 '_oh': {},
 '_dh': ['/content'],
 'In': ['',
  '# начало scope\n# global namespace\nx = "модуль"\n\n\ndef f() -> None:\n    # начало scope\n    x = "локальная"\n    # достали x из local namespace\n    print("в 

`local namespace`:
- Что это: namespace текущего вызова функции/метода.
- Когда создаётся: при входе в функцию, уничтожается при выходе (если нет не является `enclosing`).

In [7]:
def f(x: int) -> None:
    y = 1
    print(locals())


f(10)

{'x': 10, 'y': 1}


In [8]:
def f(arg: int) -> None:
    print(locals())
    print(locals() == globals())


f(1)

{'arg': 1}
False


`enclosing namespace`:
- Что это: пространство имён внешней функции (и вложенных уровней), доступное для внутренних функций.
- Когда создаётся: при вызове внешней функции (или при выполнении `comprehension`).
- Время жизни: пока есть активные ссылки.


In [9]:
from collections.abc import Callable


def make_counter() -> Callable[[], int]:
    count = 0

    def step() -> int:
        nonlocal count  # связываем имя из enclosing функции
        count += 1
        return count

    return step

In [10]:
from collections.abc import Callable


x = 0


def outer() -> Callable[[], int]:
    x = 100

    def inner() -> int:
        global x  # это имя теперь относится к глобальному x, а не к outer.x
        x += 1
        return x

    return inner()


print(outer())
print(x)

1
1


- `global x` — связываем с именем из глобального пространства модуля.
- `nonlocal x` — связываем с именем во внешней функции.

In [11]:
funcs = [lambda i=i: i for i in range(3)]  # копируем i из enclosing comprehension
print([f() for f in funcs])

[0, 1, 2]


Почему в Python стоит писать функцию `main`:

In [12]:
def main() -> int:
    ...
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


SystemExit: 0

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


- `def main():` определение функции `main` — в ней размещается «стартовая» логика.
- `main()`: обычный вызов функции.
- `__name__`: специальная переменная модуля.
    - При прямом запуске файла: `__name__ == "__main__"`.
    - При импорте как модуля: `__name__ == "<имя_модуля>"`.

    `if __name__ == "__main__"`: `main()` — блок выполняется только при прямом запуске файла.

Более простая форма:

In [13]:
def main() -> None:
    ...


if __name__ == "__main__":
    main()

Почему это полезно:

- Безопасный импорт: при импорте модуля код из `main` не выполнится.
- Чистый `namespace`: временные переменные живут в локальной области видимости `main`, а не засоряют глобальное пространство модуля (`if` не создаёт отдельного `scope`).
- Тестируемость и переиспользование: `main` можно вызывать из тестов и других модулей.
- Код возврата: удобно возвращать `int` и завершать процесс через `SystemExit`.

**Замыкание** - это любая функция, которая использует переменную, определенную в области видимости (`scope`), которая является внешней по отношению к этой функции, и доступна внутри функции при вызове из области, в которой эта переменная не определена.

In [14]:
from collections.abc import Callable


def make_adder(x: int) -> Callable[[int], int]:
    def adder(y: int) -> int:
        return x + y

    return adder

In [15]:
add_two = make_adder(2)
add_five = make_adder(5)

add_two(7) + add_five(10)

24

Представьте, что перед вами стоит задача реализовать механизм пометки функций как устаревших (`deprecated`) и применить его к функции `make_adder`. Требуется реализовать функцию `deprecated` для пометки устаревших функций и применить её к `make_adder`, при этом результирующая функция обязана сохранять исходное имя `make_adder`:

In [16]:
import typing as tp
import warnings
from collections.abc import Callable


P = tp.ParamSpec("P")
R = tp.TypeVar("R")


def deprecated(func: Callable[P, R]) -> Callable[P, R]:
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        warnings.warn(f"Warning: {func.__name__} is deprecated", DeprecationWarning)
        return func(*args, **kwargs)

    return wrapper

In [17]:
import typing as tp
from collections.abc import Callable


def make_adder(x: int) -> Callable[[int], int]:
    """Returns a function that adds x"""

    def adder(y: int) -> int:
        return x + y

    return adder


make_adder = deprecated(make_adder)

In [18]:
add_two = make_adder(2)
add_five = make_adder(5)

add_two(7) + add_five(10)

/tmp/ipykernel_1635/1347808862.py:12: DeprecationWarning: Warning: make_adder is deprecated
  warnings.warn(f"Warning: {func.__name__} is deprecated", DeprecationWarning)


24

Мы сделали декоратор: он берёт функцию, добавляет к её вызову нужную логику и возвращает новую обёртку, которую можно вызывать так же, как оригинал.

### Декораторы

В Python предусмотрен синтаксический сахар для применения декораторов: конструкция `@deprecated`, размещённая непосредственно перед определением функции `make_adder`, эквивалентна присваиванию `make_adder = deprecated(make_adder)`, выполняемому сразу после оператора `def`.


In [19]:
import typing as tp
import warnings
from collections.abc import Callable


P = tp.ParamSpec("P")
R = tp.TypeVar("R")


def deprecated(func: Callable[P, R]) -> Callable[P, R]:
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        warnings.warn(f"Warning: {func.__name__} is deprecated", DeprecationWarning)
        return func(*args, **kwargs)

    return wrapper


@deprecated
def make_adder(x: int) -> Callable[[int], int]:
    """Returns a function that adds x"""

    def adder(y: int) -> int:
        return x + y

    return adder

В целом всё выполнено корректно; однако остаётся одно существенное замечание:

In [20]:
print(make_adder.__name__)
print(make_adder.__doc__)

wrapper
None


Для сохранения метаданных изначальной функции предлагается следущее решение:

In [21]:
import typing as tp
import warnings
from collections.abc import Callable


P = tp.ParamSpec("P")
R = tp.TypeVar("R")


def deprecated(func: Callable[P, R]) -> Callable[P, R]:
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        warnings.warn(f"Warning: {func.__name__} is deprecated", DeprecationWarning)
        return func(*args, **kwargs)

    wrapper.__name__ = func.__name__
    wrapper.__doc__ = func.__doc__
    wrapper.__module__ = func.__module__
    return wrapper


@deprecated
def make_adder(x: int) -> Callable[[int], int]:
    """Returns a function that adds x"""

    def adder(y: int) -> int:
        return x + y

    return adder


print(make_adder.__name__)
print(make_adder.__doc__)

make_adder
Returns a function that adds x


Предложенное решение корректно; однако более идиоматичным считается использование `functools.wraps` для сохранения метаданных оригинальной функции:

In [22]:
import functools
import typing as tp
import warnings
from collections.abc import Callable


P = tp.ParamSpec("P")
R = tp.TypeVar("R")


def deprecated(func: Callable[P, R]) -> Callable[P, R]:
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        warnings.warn(f"Warning: {func.__name__} is deprecated", DeprecationWarning)
        return func(*args, **kwargs)

    return wrapper


@deprecated
def make_adder(x: int) -> Callable[[int], int]:
    """Returns a function that adds x"""

    def adder(y: int) -> int:
        return x + y

    return adder


print(make_adder.__name__)
print(make_adder.__doc__)

make_adder
Returns a function that adds x


Хорошо, мы умеем создавать декораторы, а теперь реализуем параметризованный декоратор `deprecated`, принимающий аргумент даты начала устаревания (например, `since`) и включающий эту дату в предупреждение при каждом вызове помеченной функции, чтобы явно указывать, с какого момента функциональность считается устаревшей.

In [23]:
import functools
import warnings
from collections.abc import Callable
from datetime import date, datetime


P = tp.ParamSpec("P")
R = tp.TypeVar("R")


def deprecated(
    *, since: date | datetime | None = None
) -> Callable[[Callable[P, R]], Callable[P, R]]:
    def decorator(func: Callable[P, R]) -> Callable[P, R]:
        @functools.wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
            parts: list[str] = [f"Function {func.__name__}() is deprecated."]
            if since is not None:
                stamp = (
                    since.date().isoformat()
                    if isinstance(since, datetime)
                    else since.isoformat()
                )
                parts.append(f"Since {stamp}.")
            warnings.warn(" ".join(parts), DeprecationWarning)
            return func(*args, **kwargs)

        return wrapper

    return decorator


# decorator = deprecated(since=date(2024, 9, 1))
# make_adder = decorator(make_adder)
@deprecated(since=date(2024, 9, 1))
def make_adder(x: int) -> Callable[[int], int]:
    """Returns a function that adds x"""

    def adder(y: int) -> int:
        return x + y

    return adder


add_two = make_adder(2)
add_five = make_adder(5)

add_two(7) + add_five(10)

/tmp/ipykernel_1635/2623639632.py:25: DeprecationWarning: Function make_adder() is deprecated. Since 2024-09-01.
  warnings.warn(" ".join(parts), DeprecationWarning)


24

Главная идея: вы превратили "простой декоратор" в "параметризованный декоратор". Для этого добавили один внешний уровень вложенности. Чтобы декоратор мог принимать аргументы (`since`). Внешняя функция `deprecated(...)` получает параметры и возвращает "настоящий" декоратор `decorator`, который уже умеет оборачивать функцию.

Помимо атрибутов у классов и их экземпляров, атрибуты есть и у функций ([PEP 232](https://peps.python.org/pep-0232)). Это позволяет хранить состояние прямо на объекте функции. Давайте реализуем декоратор `count_calls`, который считает количество вызовов помеченной функции. Состояние будем хранить в атрибуте функции (например, `func.calls`).

In [24]:
import functools
import typing as tp
from collections.abc import Callable


P = tp.ParamSpec("P")
R = tp.TypeVar("R")


def count_calls(func: Callable[P, R]) -> Callable[P, R]:
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        wrapper.calls += 1
        return func(*args, **kwargs)

    wrapper.calls = 0
    return wrapper


@count_calls
def add(a: int, b: int) -> int:
    return a + b


add(1, 2)
add(3, 4)
print(add.calls)

2


Класс в Python становится вызываемым при определении метода `__call__`. Следовательно, его можно использовать как декоратор: достаточно реализовать `__init__` (для приёма декорируемой функции или параметров) и `__call__` (для перехвата и обработки вызова). Предлагается создать класс‑декоратор для кэширования: при первом вызове вычислять результат и сохранять его по ключу, сформированному из аргументов; при последующих вызовах возвращать сохранённое значение. Это позволяет ускорять функции без изменения их интерфейса.

In [25]:
import random
import typing as tp
from collections import deque
from collections.abc import Callable


P = tp.ParamSpec("P")
R = tp.TypeVar("R")


class Memoized(tp.Generic[P, R]):
    def __init__(self, cache_size: int = 100) -> None:
        self.cache_size: int = cache_size
        self.call_args_queue: deque[int] = deque()
        self.call_args_to_result: dict[int, R] = {}

    def __call__(self, fn: Callable[P, R]) -> Callable[P, R]:
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
            memoization_key: int = self._convert_call_arguments_to_hash(args, kwargs)
            if memoization_key not in self.call_args_to_result:
                result: R = fn(*args, **kwargs)
                self._update_cache_key_with_value(memoization_key, result)
                self._evict_cache_if_necessary()
            return self.call_args_to_result[memoization_key]

        return wrapper

    def _update_cache_key_with_value(self, key: int, value: R) -> None:
        self.call_args_to_result[key] = value
        self.call_args_queue.append(key)

    def _evict_cache_if_necessary(self) -> None:
        if len(self.call_args_queue) > self.cache_size:
            oldest_key: int = self.call_args_queue.popleft()
            del self.call_args_to_result[oldest_key]

    @staticmethod
    def _convert_call_arguments_to_hash(
        args: tuple[tp.Any, ...],
        kwargs: dict[str, tp.Any],
    ) -> int:
        return hash(str(args) + str(kwargs))


@Memoized(cache_size=5)
def get_not_so_random_number_with_max(max_value: float) -> float:
    return random.random() * max_value


In [26]:
print(get_not_so_random_number_with_max(100))
print(get_not_so_random_number_with_max(100))

2.0213383648854166
2.0213383648854166


In [27]:
print(get_not_so_random_number_with_max(1))
print(get_not_so_random_number_with_max(2))
print(get_not_so_random_number_with_max(3))
print(get_not_so_random_number_with_max(4))
print(get_not_so_random_number_with_max(5))

0.895382498567677
1.602136785328895
0.7519045650706723
0.9926471304120157
0.9947779118186639


In [28]:
print(get_not_so_random_number_with_max(100))

97.64234577068692


Можно сочетать несколько декораторов: Python применяет их снизу вверх. Запись
```python
@A
@B
def f(...):
```
эквивалентна `f = A(B(f))`. Порядок важен: где считать вызовы и где показывать предупреждение, зависит от расположения декораторов.

In [29]:
@deprecated(since=date(2024, 9, 1))
@Memoized(cache_size=5)
@count_calls
def slow_add(a: int, b: int) -> int:
    return a + b

In [30]:
slow_add(1, 2)

/tmp/ipykernel_1635/2623639632.py:25: DeprecationWarning: Function wrapper() is deprecated. Since 2024-09-01.
  warnings.warn(" ".join(parts), DeprecationWarning)


3

In [31]:
slow_add(1, 2)

/tmp/ipykernel_1635/2623639632.py:25: DeprecationWarning: Function wrapper() is deprecated. Since 2024-09-01.
  warnings.warn(" ".join(parts), DeprecationWarning)


3

In [32]:
slow_add(3, 4)

/tmp/ipykernel_1635/2623639632.py:25: DeprecationWarning: Function wrapper() is deprecated. Since 2024-09-01.
  warnings.warn(" ".join(parts), DeprecationWarning)


7

In [33]:
slow_add.calls

AttributeError: 'function' object has no attribute 'calls'

In [34]:
@count_calls
@deprecated(since=date(2024, 9, 1))
@Memoized(cache_size=5)
def slow_add(a: int, b: int) -> int:
    return a + b

In [35]:
slow_add(3, 4)

/tmp/ipykernel_1635/2623639632.py:25: DeprecationWarning: Function wrapper() is deprecated. Since 2024-09-01.
  warnings.warn(" ".join(parts), DeprecationWarning)


7

In [36]:
slow_add(3, 4)

/tmp/ipykernel_1635/2623639632.py:25: DeprecationWarning: Function wrapper() is deprecated. Since 2024-09-01.
  warnings.warn(" ".join(parts), DeprecationWarning)


7

In [37]:
slow_add.calls

2

In [38]:
slow_add(3, 5)

/tmp/ipykernel_1635/2623639632.py:25: DeprecationWarning: Function wrapper() is deprecated. Since 2024-09-01.
  warnings.warn(" ".join(parts), DeprecationWarning)


8

In [39]:
slow_add.calls

3

In [40]:
@count_calls
@Memoized(cache_size=5)
@deprecated(since=date(2024, 9, 1))
def slow_add(a: int, b: int) -> int:
    return a + b

In [41]:
slow_add(3, 5)

/tmp/ipykernel_1635/2623639632.py:25: DeprecationWarning: Function slow_add() is deprecated. Since 2024-09-01.
  warnings.warn(" ".join(parts), DeprecationWarning)


8

In [42]:
slow_add(3, 5)

8

Немного про встроенные полезные встроенные декораторы:

Вы уже использовали:
- `@staticmethod`, `@classmethod`, `@property`
- `@functools.wraps`
- `@dataclasses.dataclass`

Рассмотрим те, которые вы еще не использовали:

`contextlib.contextmanager` — удобный способ создавать менеджеры контекста

In [43]:
import typing as tp
from contextlib import contextmanager


@contextmanager
def magic_contextmanager() -> tp.Iterator[int]:
    print('before')
    yield 42
    print('after')

with magic_contextmanager() as r:
    print(f'got {r}')

with magic_contextmanager() as r:
    raise RuntimeError('Oops')

before
got 42
after
before


RuntimeError: Oops

- `@contextmanager` из модуля `contextlib` позволяет писать менеджеры контекста как обычную функцию‑генератор с `yield`.
- Всё, что стоит до `yield`, выполняется при входе в `with` (на стадии `__enter__`).
- Значение, которое отдаёт `yield`, попадает в переменную после `as` (`r` в примере).
- Всё, что стоит после `yield`, выполняется при выходе из `with` (на стадии `exit`). Но чтобы этот хвост точно отработал даже при ошибках внутри блока, вокруг `yield` обычно ставят `try/finally`.

In [44]:
from contextlib import contextmanager


@contextmanager
def magic_contextmanager() -> tp.Iterator[int]:
    print("before")
    try:
        yield 42
    finally:
        print("after")


with magic_contextmanager() as r:
    print(f"got {r}")


with magic_contextmanager() as r:
    raise RuntimeError("Oops")

before
got 42
after
before
after


RuntimeError: Oops

In [45]:
from contextlib import contextmanager


@contextmanager
def magic_contextmanager() -> tp.Iterator[int]:
    print("before")
    try:
        yield 42
    except Exception as e:
        print(f"{e=}")
    finally:
        print("after")


with magic_contextmanager() as r:
    print(f"got {r}")


with magic_contextmanager() as r:
    raise RuntimeError("Oops")

before
got 42
after
before
e=RuntimeError('Oops')
after


- `functools.lru_cache` - LRU cache над функцией. Декоратор, который запоминает (кэширует) результаты функции для уже встречавшихся аргументов. LRU = Least Recently Used: при переполнении кэша удаляются давно не использованные записи.
- `functools.cache` (python 3.9+)

```python
def cache(user_function, /):
    'Simple lightweight unbounded cache.  Sometimes called "memoize".'
    return lru_cache(maxsize=None)(user_function)
```

In [46]:
from functools import lru_cache


@lru_cache(maxsize=128)
def fib(n: int) -> int:
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(f"{fib(123)=}")

fib(123)=22698374052006863956975682


- `maxsize`: размер кэша.
    - Число (например, 128) — ограниченный LRU-кэш.
    - `None` — без ограничения; эквивалентно декоратору `functools.cache`.
- `typed: False` по умолчанию. Если `True`, различает `1` и `1.0`, `'1'` и т. п.


Как работает:
- Ключ кэша = кортеж из позиционных и именованных аргументов (они должны быть хешируемыми).
- Если в кэше есть запись для таких аргументов — функция не выполняется, сразу возвращается сохранённый результат (hit).
- Если записи нет — функция выполняется, результат кладётся в кэш (miss).
- Когда достигнут `maxsize`, удаляются самые давно неиспользованные записи.
